# Complete Nuclear Receptor Ligand Analysis by Family
## Step 2: Molecular Property Analysis + Step 3: Chemical Space Analysis
### Applied to ALL NR Families (Steroid, Non-steroid, Orphan)

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors, QED
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import FilterCatalog
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import umap
import os
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 300

print("✅ All imports successful!")

## Load Data and Check Families

In [ ]:
# Load the dataset
input_file = "/mnt/user-data/uploads/ligands_with_final_chemical_classification.csv"
df_all = pd.read_csv(input_file)

print(f"✅ Loaded {len(df_all)} total ligands")
print(f"\n📊 NR Family Distribution:")
family_counts = df_all["NR_Family"].value_counts()
print(family_counts)
print(f"\nTotal families: {len(family_counts)}")

# Store family names for iteration
families = df_all["NR_Family"].unique()
print(f"\nFamilies to analyze: {list(families)}")

## Step 2: Molecular Property Analysis Functions

In [ ]:
# Initialize PAINS filter
params = FilterCatalog.FilterCatalogParams()
params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
pains_catalog = FilterCatalog.FilterCatalog(params)

def calculate_full_descriptors(smiles):
    """Calculate comprehensive molecular descriptors including PAINS filtering"""
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        return {col: None for col in [
            "MolWt", "TPSA", "NumHDonors", "NumHAcceptors", "LogP",
            "NumRotatableBonds", "QED", "RingCount", "AromaticRingCount",
            "StereoCenterCount", "NumHeteroatoms", "FractionCSP3", "MolMR",
            "PAINS_Flag", "Lipinski_Flag", "Veber_Flag"
        ]}
    
    # Basic & Physicochemical properties
    mol_wt = Descriptors.MolWt(mol)
    tpsa = Descriptors.TPSA(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)
    logp = Crippen.MolLogP(mol)
    
    # Structural complexity
    rot_bonds = Lipinski.NumRotatableBonds(mol)
    qed = QED.qed(mol)
    ring_count = Descriptors.RingCount(mol)
    aromatic_rings = rdMolDescriptors.CalcNumAromaticRings(mol)
    stereo_centers = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True))
    num_heteroatoms = Lipinski.NumHeteroatoms(mol)
    fraction_csp3 = Lipinski.FractionCSP3(mol)
    mol_mr = Crippen.MolMR(mol)
    
    # PAINS filtering
    pains_matches = pains_catalog.GetMatches(mol)
    pains_flag = "Fail" if pains_matches else "Pass"
    
    # Lipinski's Rule of Five
    lipinski_pass = (
        mol_wt <= 500 and
        logp <= 5 and
        hbd <= 5 and
        hba <= 10
    )
    lipinski_flag = "Pass" if lipinski_pass else "Fail"
    
    # Veber's Rule
    veber_pass = (rot_bonds <= 10 and tpsa <= 140)
    veber_flag = "Pass" if veber_pass else "Fail"
    
    return {
        "MolWt": mol_wt,
        "TPSA": tpsa,
        "NumHDonors": hbd,
        "NumHAcceptors": hba,
        "LogP": logp,
        "NumRotatableBonds": rot_bonds,
        "QED": qed,
        "RingCount": ring_count,
        "AromaticRingCount": aromatic_rings,
        "StereoCenterCount": stereo_centers,
        "NumHeteroatoms": num_heteroatoms,
        "FractionCSP3": fraction_csp3,
        "MolMR": mol_mr,
        "PAINS_Flag": pains_flag,
        "Lipinski_Flag": lipinski_flag,
        "Veber_Flag": veber_flag
    }

print("✅ Descriptor calculation function defined")

## Process Each Family - Step 2: Molecular Properties

In [ ]:
# Create output directory
os.makedirs("family_analysis_results", exist_ok=True)

# Dictionary to store processed dataframes
family_dfs = {}

print("\n" + "="*80)
print("STEP 2: MOLECULAR PROPERTY ANALYSIS")
print("="*80 + "\n")

for family in families:
    print(f"\n📊 Processing {family} family...")
    
    # Filter for this family
    family_df = df_all[df_all["NR_Family"] == family].copy()
    print(f"   Ligands: {len(family_df)}")
    
    # Calculate descriptors
    descriptor_data = []
    for idx, row in family_df.iterrows():
        smiles = row.get("SMILES", "")
        props = calculate_full_descriptors(smiles)
        descriptor_data.append(props)
    
    # Add descriptors as new columns
    desc_df = pd.DataFrame(descriptor_data)
    family_df = pd.concat([family_df.reset_index(drop=True), desc_df.reset_index(drop=True)], axis=1)
    
    # Save to CSV
    family_clean = family.lower().replace(" ", "_").replace("-", "_")
    output_file = f"family_analysis_results/{family_clean}_with_full_descriptors.csv"
    family_df.to_csv(output_file, index=False)
    
    family_dfs[family] = family_df
    
    print(f"   ✅ Saved: {output_file}")
    print(f"   ✅ Columns: {len(family_df.columns)}")

print("\n" + "="*80)
print("✅ STEP 2 COMPLETE FOR ALL FAMILIES")
print("="*80)

## Step 3: Chemical Space Analysis - For Each Family

### Define Analysis Functions

In [ ]:
def get_scaffold(smiles):
    """Extract Murcko scaffold from SMILES"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            scaffold = MurckoScaffold.GetScaffoldForMol(mol)
            return Chem.MolToSmiles(scaffold)
        return None
    except:
        return None

def analyze_family_chemical_space(family_df, family_name):
    """Complete chemical space analysis for one family"""
    
    family_clean = family_name.lower().replace(" ", "_").replace("-", "_")
    output_dir = f"family_analysis_results/{family_clean}_analysis"
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\n{'='*80}")
    print(f"ANALYZING: {family_name} ({len(family_df)} ligands)")
    print(f"{'='*80}")
    
    # Select numeric descriptor columns
    descriptor_cols = [
        "MolWt", "TPSA", "NumHDonors", "NumHAcceptors", "LogP",
        "NumRotatableBonds", "QED", "RingCount", "AromaticRingCount",
        "StereoCenterCount"
    ]
    
    df_desc = family_df[descriptor_cols].copy()
    
    # ===== FIGURE 1: Correlation Heatmap =====
    print("\n📊 Creating Figure 1: Correlation Heatmap...")
    plt.figure(figsize=(12, 10))
    corr_matrix = df_desc.corr()
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0,
                square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title(f"Correlation Heatmap of {family_name} Descriptors", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/figure1_correlation_heatmap.png", dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Saved: {output_dir}/figure1_correlation_heatmap.png")
    
    # ===== FIGURE 2: Histograms =====
    print("\n📊 Creating Figure 2: Descriptor Histograms...")
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.flatten()
    
    for idx, col in enumerate(descriptor_cols):
        if idx < len(axes):
            axes[idx].hist(df_desc[col].dropna(), bins=25, color='skyblue', edgecolor='black', alpha=0.7)
            axes[idx].set_title(f"Distribution of {col}")
            axes[idx].set_xlabel(col)
            axes[idx].set_ylabel("Count")
            
            # Add KDE
            try:
                df_desc[col].dropna().plot(kind='kde', ax=axes[idx], secondary_y=True, color='red', linewidth=2)
            except:
                pass
    
    # Hide unused subplots
    for idx in range(len(descriptor_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle(f"{family_name} Ligand Descriptors - Distributions", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/figure2_histograms.png", dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Saved: {output_dir}/figure2_histograms.png")
    
    # ===== FIGURE 3: PCA Analysis =====
    print("\n📊 Creating Figure 3: PCA Analysis...")
    
    # Drop rows with NaN
    df_clean = df_desc.dropna()
    family_df_clean = family_df.loc[df_clean.index].copy()
    
    print(f"   Dropped {len(df_desc) - len(df_clean)} rows with NaN values")
    print(f"   Clean dataset: {len(df_clean)} ligands")
    
    if len(df_clean) > 0:
        # Standardize
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(df_clean)
        
        # PCA
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)
        
        # Compute combined drug-likeness score
        family_df_clean['Lipinski_Score'] = family_df_clean['Lipinski_Flag'].map({'Pass': 1, 'Fail': 0})
        family_df_clean['Veber_Score'] = family_df_clean['Veber_Flag'].map({'Pass': 1, 'Fail': 0})
        family_df_clean['DrugLikeness_Score'] = (
            family_df_clean['QED'] + 
            family_df_clean['Lipinski_Score'] + 
            family_df_clean['Veber_Score']
        ) / 3
        
        # Create 4 PCA plots
        fig, axes = plt.subplots(2, 2, figsize=(16, 14))
        
        # Plot 1: Colored by QED
        scatter1 = axes[0,0].scatter(X_pca[:, 0], X_pca[:, 1], 
                                     c=family_df_clean['QED'], cmap='viridis', 
                                     s=50, alpha=0.6, edgecolors='k', linewidth=0.5)
        axes[0,0].set_xlabel(f'PCA1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
        axes[0,0].set_ylabel(f'PCA2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
        axes[0,0].set_title(f'PCA of {family_name} Ligand Chemical Space Colored by QED')
        plt.colorbar(scatter1, ax=axes[0,0], label='QED')
        
        # Plot 2: Colored by Lipinski
        colors_lipinski = family_df_clean['Lipinski_Flag'].map({'Pass': 'green', 'Fail': 'red'})
        for flag, color in [('Pass', 'green'), ('Fail', 'red')]:
            mask = family_df_clean['Lipinski_Flag'] == flag
            axes[0,1].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                            c=color, label=f'Lipinski {flag}', 
                            s=50, alpha=0.6, edgecolors='k', linewidth=0.5)
        axes[0,1].set_xlabel(f'PCA1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
        axes[0,1].set_ylabel(f'PCA2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
        axes[0,1].set_title(f'PCA of {family_name} Ligand Chemical Space Colored by Lipinski Rule')
        axes[0,1].legend()
        
        # Plot 3: Colored by Veber
        for flag, color in [('Pass', 'blue'), ('Fail', 'orange')]:
            mask = family_df_clean['Veber_Flag'] == flag
            axes[1,0].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                            c=color, label=f'Veber {flag}', 
                            s=50, alpha=0.6, edgecolors='k', linewidth=0.5)
        axes[1,0].set_xlabel(f'PCA1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
        axes[1,0].set_ylabel(f'PCA2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
        axes[1,0].set_title(f'PCA of {family_name} Ligand Chemical Space Colored by Veber Rule')
        axes[1,0].legend()
        
        # Plot 4: Colored by Combined Drug-Likeness
        scatter4 = axes[1,1].scatter(X_pca[:, 0], X_pca[:, 1], 
                                     c=family_df_clean['DrugLikeness_Score'], 
                                     cmap='RdYlGn', s=50, alpha=0.6, 
                                     edgecolors='k', linewidth=0.5)
        axes[1,1].set_xlabel(f'PCA1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
        axes[1,1].set_ylabel(f'PCA2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
        axes[1,1].set_title(f'PCA of {family_name} Ligand Chemical Space Colored by Combined Drug-Likeness Score')
        plt.colorbar(scatter4, ax=axes[1,1], label='Combined Drug-Likeness')
        
        plt.tight_layout()
        plt.savefig(f"{output_dir}/figure3_pca_analysis.png", dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   ✅ Saved: {output_dir}/figure3_pca_analysis.png")
    
        # ===== FIGURE 4: t-SNE and UMAP =====
        print("\n📊 Creating Figure 4: t-SNE and UMAP clustering...")
        
        # t-SNE
        tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(df_clean)-1))
        X_tsne = tsne.fit_transform(X_scaled)
        
        # UMAP
        umap_model = umap.UMAP(n_components=2, random_state=42, n_neighbors=min(15, len(df_clean)-1))
        X_umap = umap_model.fit_transform(X_scaled)
        
        # K-Means clustering
        n_clusters = min(6, len(df_clean) // 10 + 1)
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        clusters = kmeans.fit_predict(X_scaled)
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        
        # t-SNE plot
        scatter_tsne = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], 
                                       c=clusters, cmap='tab10', 
                                       s=50, alpha=0.6, edgecolors='k', linewidth=0.5)
        axes[0].set_xlabel('tSNE1')
        axes[0].set_ylabel('tSNE2')
        axes[0].set_title(f't-SNE of {family_name} Ligands Colored by KMeans Cluster')
        plt.colorbar(scatter_tsne, ax=axes[0], label='Cluster')
        
        # UMAP plot
        scatter_umap = axes[1].scatter(X_umap[:, 0], X_umap[:, 1], 
                                       c=clusters, cmap='tab10', 
                                       s=50, alpha=0.6, edgecolors='k', linewidth=0.5)
        axes[1].set_xlabel('UMAP1')
        axes[1].set_ylabel('UMAP2')
        axes[1].set_title(f'UMAP of {family_name} Ligands Colored by KMeans Cluster')
        plt.colorbar(scatter_umap, ax=axes[1], label='Cluster')
        
        plt.tight_layout()
        plt.savefig(f"{output_dir}/figure4_tsne_umap.png", dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   ✅ Saved: {output_dir}/figure4_tsne_umap.png")
    
    # ===== FIGURE 5: Scaffold Analysis =====
    print("\n📊 Creating Figure 5: Scaffold Analysis...")
    
    family_df['Scaffold'] = family_df['SMILES'].apply(get_scaffold)
    scaffold_counts = family_df['Scaffold'].value_counts()
    
    total_ligands = len(family_df)
    unique_scaffolds = scaffold_counts.nunique()
    scaffold_diversity = unique_scaffolds / total_ligands if total_ligands > 0 else 0
    
    print(f"   Total ligands: {total_ligands}")
    print(f"   Unique scaffolds: {unique_scaffolds}")
    print(f"   Scaffold diversity: {scaffold_diversity:.3f}")
    
    # Save scaffold statistics
    scaffold_stats = pd.DataFrame({
        'Scaffold': scaffold_counts.index,
        'Count': scaffold_counts.values
    })
    scaffold_stats.to_csv(f"{output_dir}/scaffold_statistics.csv", index=False)
    print(f"   ✅ Saved: {output_dir}/scaffold_statistics.csv")
    
    # Plot top 15 scaffolds
    top_n = min(15, len(scaffold_counts))
    top_scaffolds = scaffold_counts.head(top_n)
    
    plt.figure(figsize=(12, 8))
    plt.barh(range(top_n), top_scaffolds.values, color='teal', edgecolor='black')
    plt.yticks(range(top_n), top_scaffolds.index, fontsize=8)
    plt.xlabel('Ligand Count', fontsize=12)
    plt.ylabel('Scaffold (SMILES)', fontsize=12)
    plt.title(f'Top {top_n} Scaffolds in {family_name} Ligands', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(f"{output_dir}/figure5_scaffold_analysis.png", dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Saved: {output_dir}/figure5_scaffold_analysis.png")
    
    print(f"\n✅ COMPLETE: {family_name} analysis finished!")
    print(f"   All outputs saved in: {output_dir}/")

print("✅ Analysis function defined")

### Run Complete Analysis for Each Family

In [ ]:
print("\n" + "="*80)
print("STEP 3: CHEMICAL SPACE ANALYSIS")
print("="*80)

# Run analysis for each family
for family in families:
    analyze_family_chemical_space(family_dfs[family], family)

print("\n" + "="*80)
print("✅ ALL ANALYSES COMPLETE!")
print("="*80)

## Summary Report

In [ ]:
print("\n" + "="*80)
print("📊 FINAL SUMMARY REPORT")
print("="*80 + "\n")

summary_data = []

for family in families:
    df = family_dfs[family]
    family_clean = family.lower().replace(" ", "_").replace("-", "_")
    
    summary = {
        'Family': family,
        'N_Ligands': len(df),
        'Mean_MolWt': df['MolWt'].mean(),
        'Mean_LogP': df['LogP'].mean(),
        'Mean_TPSA': df['TPSA'].mean(),
        'Mean_QED': df['QED'].mean(),
        'Lipinski_Pass_Rate_%': (df['Lipinski_Flag'] == 'Pass').mean() * 100,
        'Veber_Pass_Rate_%': (df['Veber_Flag'] == 'Pass').mean() * 100,
        'PAINS_Pass_Rate_%': (df['PAINS_Flag'] == 'Pass').mean() * 100,
        'Output_Dir': f"family_analysis_results/{family_clean}_analysis/"
    }
    summary_data.append(summary)

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv("family_analysis_results/complete_summary.csv", index=False)

print(summary_df.to_string(index=False))

print("\n" + "="*80)
print("📁 OUTPUT FILES STRUCTURE:")
print("="*80)
print("""
family_analysis_results/
├── complete_summary.csv                      # Overall summary statistics
├── steroid_with_full_descriptors.csv         # Steroid data with all descriptors
├── non_steroid_with_full_descriptors.csv     # Non-steroid data with all descriptors
├── orphan_with_full_descriptors.csv          # Orphan data with all descriptors
├── steroid_analysis/
│   ├── figure1_correlation_heatmap.png
│   ├── figure2_histograms.png
│   ├── figure3_pca_analysis.png
│   ├── figure4_tsne_umap.png
│   ├── figure5_scaffold_analysis.png
│   └── scaffold_statistics.csv
├── non_steroid_analysis/
│   └── [same structure as above]
└── orphan_analysis/
    └── [same structure as above]
""")

print("\n✅ Complete summary saved: family_analysis_results/complete_summary.csv")
print("\n🎉 ALL DONE! Check the family_analysis_results/ directory for all outputs.")